# Data Quality: Identifiers & Dates — Findings

Explore `shipment_id`, `order_date`, `ship_date`, `ship_dow`, `ship_month` in `train.csv`/`test.csv` and catalog data quality issues. Diagnostic only — nothing is dropped or modified here. Every row-drop candidate is exported to `../../../data/row_flags/flagged_rows.csv` at the end, for a later shared step to combine with everyone else's flags and apply.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

ID_DATE_COLS = ["shipment_id", "order_date", "ship_date", "ship_dow", "ship_month"]

train = pd.read_csv("../../../data/train.csv")
test = pd.read_csv("../../../data/test.csv")

df = train[ID_DATE_COLS].copy()
print(train.shape, test.shape)
df.head()

(40030, 33) (10120, 31)


,shipment_id,order_date,ship_date,ship_dow,ship_month
0,SHP-42-0003885,2024-04-23,2024-04-23,1,4
1,SHP-42-0010239,2025-01-07,2025-01-09,3,1
2,SHP-42-0035486,2025-07-30,2025-07-30,2,7
3,SHP-42-0008180,2025-03-24,2025-03-26,2,3
4,SHP-42-0015190,2025-05-06,2025-05-08,3,5


## 1. `shipment_id` — duplicates and format

In [2]:
# Nulls and format
print("nulls:", df["shipment_id"].isnull().sum())

id_pattern = r"^SHP-\d+-\d{7}$"
bad_format = df.loc[~df["shipment_id"].str.match(id_pattern)]
print("rows not matching SHP-##-####### pattern:", len(bad_format))
bad_format.head()

nulls: 0
rows not matching SHP-##-####### pattern: 0


,shipment_id,order_date,ship_date,ship_dow,ship_month


In [3]:
# Deeper format check: is the "SHP" prefix, the middle segment, and the
# numeric-id width actually constant, or just coincidentally matching the regex?
parts = df["shipment_id"].str.split("-", expand=True)
print("prefix values:", parts[0].unique())
print("middle-segment values:", parts[1].unique())
print("numeric-segment digit-length values:", parts[2].str.len().unique())
print("numeric-segment range:", parts[2].astype(int).min(), "-", parts[2].astype(int).max())
print("any case variation (raw != upper):", (df["shipment_id"] != df["shipment_id"].str.upper()).sum())
print("any leading/trailing whitespace:", (df["shipment_id"] != df["shipment_id"].str.strip()).sum())

prefix values: ['SHP']
middle-segment values: ['42']
numeric-segment digit-length values: [7]
numeric-segment range: 0 - 49999
any case variation (raw != upper): 0
any leading/trailing whitespace: 0


In [4]:
# Duplicate rows on shipment_id
dup_ids = df.loc[df["shipment_id"].duplicated(keep=False), "shipment_id"].unique()
print(f"total rows: {len(train)}, unique shipment_id: {train['shipment_id'].nunique()}")
print(f"shipment_ids appearing more than once: {len(dup_ids)}")

dup_rows = train[train["shipment_id"].isin(dup_ids)].sort_values("shipment_id")
print("rows involved in duplication:", len(dup_rows))
dup_rows.head(10)

total rows: 40030, unique shipment_id: 39949
shipment_ids appearing more than once: 81
rows involved in duplication: 162


,shipment_id,order_date,ship_date,ship_dow,ship_month,shipment_mode,carrier,service_level,origin_metro,dest_metro,origin_zip,dest_zip,origin_lat,origin_lon,dest_lat,dest_lon,distance_miles,zone,weight_lbs,length_in,width_in,height_in,pallet_count,residential_flag,signature_required,declared_value_usd,fuel_surcharge_pct,weather_severity,carrier_congestion_index,promised_days,carrier_cost_usd,transit_days,delivered_on_time
14248,SHP-42-0000070,2025-01-08,2025-01-08,2,1,ltl_freight,Vela Freight,LTL Guaranteed,Chicago IL,Los Angeles CA,60654,90084,41.88,-87.63,34.05,-118.24,1742.2,7,481.14,33.6,28.2,22.9,1,0,0,0.00,0.1428,0.290,0.099,7,397.61,7,1
6080,SHP-42-0000070,2025-01-08,2025-01-08,2,1,ltl_freight,Vela Freight,LTL Guaranteed,Chicago IL,Los Angeles CA,60654,90084,41.88,-87.63,34.05,-118.24,1742.2,7,481.14,33.6,28.2,22.9,1,0,0,0.00,0.1428,0.290,0.099,7,397.61,7,1
26450,SHP-42-0000422,2025-08-27,2025-08-28,3,8,parcel,Northstar Parcel,Overnight,Los Angeles CA,Nashville TN,90025,37281,34.05,-118.24,36.16,-86.78,1776.5,7,6.76,13.5,9.6,7.3,0,0,0,8.47,0.0998,0.096,0.096,2,30.19,1,1
18890,SHP-42-0000422,2025-08-27,2025-08-28,3,8,parcel,Northstar Parcel,Overnight,Los Angeles CA,Nashville TN,90025,37281,34.05,-118.24,36.16,-86.78,1776.5,7,6.76,13.5,9.6,7.3,0,0,0,8.47,0.0998,0.096,0.096,2,30.19,1,1
35903,SHP-42-0001447,2024-02-29,2024-03-01,4,3,ltl_freight,Ironclad Freight,LTL Standard,Miami FL,Columbus OH,33154,43224,25.76,-80.19,39.96,-82.99,994.3,5,NaN,43.9,40.3,NaN,1,0,0,32.44,0.0642,0.211,0.128,9,244.48,11,0
14884,SHP-42-0001447,2024-02-29,2024-03-01,4,3,ltl_freight,Ironclad Freight,LTL Standard,Miami FL,Columbus OH,33154,43224,25.76,-80.19,39.96,-82.99,994.3,5,NaN,43.9,40.3,NaN,1,0,0,32.44,0.0642,0.211,0.128,9,244.48,11,0
34176,SHP-42-0002036,2024-02-22,2024-02-22,3,2,ltl_freight,Vela Freight,LTL Standard,Salt Lake City UT,Nashville TN,84120,37207,40.76,-111.89,36.16,-86.78,1390.1,6,660.28,40.0,30.4,30.0,1,1,0,0.00,0.1637,0.588,0.038,10,368.34,9,1
30831,SHP-42-0002036,2024-02-22,2024-02-22,3,2,ltl_freight,Vela Freight,LTL Standard,Salt Lake City UT,Nashville TN,84120,37207,40.76,-111.89,36.16,-86.78,1390.1,6,660.28,40.0,30.4,30.0,1,1,0,0.00,0.1637,0.588,0.038,10,368.34,9,1
10481,SHP-42-0002091,2024-05-11,2024-05-11,5,5,parcel,Cobalt Regional,Expedited,Portland OR,Seattle WA,97285,98186,45.51,-122.68,47.61,-122.33,146.0,2,5.12,6.9,6.8,4.8,0,0,0,0.00,0.1052,0.164,0.196,3,12.44,2,1
36143,SHP-42-0002091,2024-05-11,2024-05-11,5,5,parcel,Cobalt Regional,Expedited,Portland OR,Seattle WA,97285,98186,45.51,-122.68,47.61,-122.33,146.0,2,5.12,6.9,6.8,4.8,0,0,0,0.00,0.1052,0.164,0.196,3,12.44,2,1


In [5]:
# Are the duplicated rows byte-for-byte identical (all 33 columns), or do other
# columns disagree between the two occurrences of the same id?
exact_dupe_mask = train.duplicated(keep=False) & train["shipment_id"].isin(dup_ids)
print("rows that are exact full-row duplicates:", exact_dupe_mask.sum())
print("ids covered by exact duplicates:", train.loc[exact_dupe_mask, "shipment_id"].nunique())
print("ids where duplication is NOT a clean exact copy (conflicting values):",
      len(set(dup_ids) - set(train.loc[exact_dupe_mask, "shipment_id"])))

rows that are exact full-row duplicates: 162
ids covered by exact duplicates: 81
ids where duplication is NOT a clean exact copy (conflicting values): 0


### `shipment_id` across train/test

`test.csv` should be a disjoint holdout. Check for internal duplicates in `test.csv` and any id overlap with `train.csv`.

In [6]:
print("test rows:", len(test), "unique test ids:", test["shipment_id"].nunique())
print("duplicate ids within test.csv:", test["shipment_id"].duplicated().sum())

overlap_ids = set(train["shipment_id"]) & set(test["shipment_id"])
print("shipment_ids present in BOTH train.csv and test.csv:", len(overlap_ids))

common_cols = [c for c in test.columns if c in train.columns]
sample_overlap = sorted(overlap_ids)[:3]
train.loc[train["shipment_id"].isin(sample_overlap), common_cols]

test rows: 10120 unique test ids: 10114
duplicate ids within test.csv: 6
shipment_ids present in BOTH train.csv and test.csv: 63


,shipment_id,order_date,ship_date,ship_dow,ship_month,shipment_mode,carrier,service_level,origin_metro,dest_metro,origin_zip,dest_zip,origin_lat,origin_lon,dest_lat,dest_lon,distance_miles,zone,weight_lbs,length_in,width_in,height_in,pallet_count,residential_flag,signature_required,declared_value_usd,fuel_surcharge_pct,weather_severity,carrier_congestion_index,promised_days,carrier_cost_usd
6226,SHP-42-0001786,2024-12-09,2024-12-11,2,12,ltl_freight,Vela Freight,LTL Standard,Atlanta GA,Charlotte NC,30362,28272,33.75,-84.39,35.23,-80.84,226.5,3,631.69,48.3,33.2,39.7,1,0,0,0.00,0.1509,0.388,0.832,7,198.47
19834,SHP-42-0002905,2024-09-03,2024-09-04,2,9,ltl_freight,Vela Freight,LTL Guaranteed,Phoenix AZ,Portland OR,85029,97224,33.45,-112.07,45.51,-122.68,1005.3,6,451.83,50.2,35.4,36.2,1,1,0,43.94,0.1446,0.074,0.137,7,290.54
28312,SHP-42-0000973,2024-07-27,2024-07-27,5,7,parcel,Northstar Parcel,Ground,Columbus OH,Boston MA,43277,2111,39.96,-82.99,42.36,-71.06,641.7,5,3.80,5.7,7.5,5.9,0,0,0,28.24,0.0924,0.404,0.189,6,11.46


In [7]:
# The 6 duplicate ids within test.csv itself -- are they exact full-row duplicates
# too, same as train's?
test_dup_ids = test.loc[test["shipment_id"].duplicated(keep=False), "shipment_id"].unique()
test_exact_dupe_mask = test.duplicated(keep=False) & test["shipment_id"].isin(test_dup_ids)
print("test.csv duplicate ids:", len(test_dup_ids))
print("test.csv rows that are exact full-row duplicates:", test_exact_dupe_mask.sum())
print("test.csv ids covered by exact duplicates:", test.loc[test_exact_dupe_mask, "shipment_id"].nunique())

test.loc[test["shipment_id"].isin(test_dup_ids)].sort_values("shipment_id")[["shipment_id", "order_date", "ship_date"]]

test.csv duplicate ids: 6
test.csv rows that are exact full-row duplicates: 12
test.csv ids covered by exact duplicates: 6


,shipment_id,order_date,ship_date
2023,SHP-42-0003943,2024-06-23,2024-06-24
10108,SHP-42-0003943,2024-06-23,2024-06-24
5505,SHP-42-0013868,2024-06-27,2024-06-28
8763,SHP-42-0013868,2024-06-27,2024-06-28
1356,SHP-42-0018552,2025-04-04,2025-04-05
3382,SHP-42-0018552,2025-04-04,2025-04-05
88,SHP-42-0034654,2024-01-01,2024-01-03
7434,SHP-42-0034654,2024-01-01,2024-01-03
6277,SHP-42-0038513,2024-10-12,2024-10-15
10006,SHP-42-0038513,2024-10-12,2024-10-15


## 2. `order_date` / `ship_date` — parseability, ordering, range

In [8]:
order_date = pd.to_datetime(df["order_date"], errors="coerce")
ship_date = pd.to_datetime(df["ship_date"], errors="coerce")

print("order_date unparseable:", order_date.isnull().sum())
print("ship_date unparseable:", ship_date.isnull().sum())
print("order_date range:", order_date.min(), "->", order_date.max())
print("ship_date range:", ship_date.min(), "->", ship_date.max())

today = pd.Timestamp.today().normalize()
print("order_date in the future:", (order_date > today).sum())
print("ship_date in the future:", (ship_date > today).sum())

order_date unparseable: 0
ship_date unparseable: 0
order_date range: 2023-12-29 00:00:00 -> 2025-12-30 00:00:00
ship_date range: 2024-01-01 00:00:00 -> 2025-12-30 00:00:00
order_date in the future: 0
ship_date in the future: 0


In [9]:
# Raw string format check, before any parsing: are both date columns consistently
# "YYYY-MM-DD" (fixed width, no stray whitespace, no alternate formats mixed in)?
for col in ["order_date", "ship_date"]:
    raw = df[col].astype(str)
    print(f"--- {col} ---")
    print("string length values:", raw.str.len().unique())
    print("rows not matching ^\\d{4}-\\d{2}-\\d{2}$:", (~raw.str.match(r"^\d{4}-\d{2}-\d{2}$")).sum())
    print("rows with leading/trailing whitespace:", (raw != raw.str.strip()).sum())

--- order_date ---
string length values: [10]
rows not matching ^\d{4}-\d{2}-\d{2}$: 0
rows with leading/trailing whitespace: 0
--- ship_date ---
string length values: [10]
rows not matching ^\d{4}-\d{2}-\d{2}$: 0
rows with leading/trailing whitespace: 0


In [10]:
# Logical ordering: a shipment can't ship before it was ordered
ship_before_order = ship_date < order_date
print("rows where ship_date < order_date:", ship_before_order.sum())
df.loc[ship_before_order]

rows where ship_date < order_date: 0


,shipment_id,order_date,ship_date,ship_dow,ship_month


In [11]:
# Order-to-ship lag distribution — look for outliers (e.g. suspiciously long holds)
lag_days = (ship_date - order_date).dt.days
print(lag_days.describe())
lag_days.value_counts().sort_index()

count    40030.000000
mean         0.997527
std          1.002068
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max          7.000000
dtype: float64


0    14753
1    14824
2     7229
3     2437
4      622
5      141
6       23
7        1
Name: count, dtype: int64

## 3. `ship_dow` / `ship_month` — derived-field consistency

These are supposed to be derived directly from `ship_date` (0 = Monday, month 1-12). Recompute them and check for mismatches, plus check the raw value ranges are valid.

In [12]:
# Valid ranges
print("ship_dow out of [0, 6]:", (~df["ship_dow"].between(0, 6)).sum())
print("ship_month out of [1, 12]:", (~df["ship_month"].between(1, 12)).sum())

# Recompute from ship_date (pandas: Monday=0, matching the stated convention) and diff
computed_dow = ship_date.dt.dayofweek
computed_month = ship_date.dt.month

dow_mismatch = (computed_dow != df["ship_dow"]) & ship_date.notnull()
month_mismatch = (computed_month != df["ship_month"]) & ship_date.notnull()

print("ship_dow disagrees with ship_date:", dow_mismatch.sum())
print("ship_month disagrees with ship_date:", month_mismatch.sum())

df.loc[dow_mismatch | month_mismatch]

ship_dow out of [0, 6]: 0
ship_month out of [1, 12]: 0
ship_dow disagrees with ship_date: 0
ship_month disagrees with ship_date: 0


,shipment_id,order_date,ship_date,ship_dow,ship_month


## 4. Same date checks, applied to `test.csv`

`test.csv` withholds the targets but still has `order_date`/`ship_date`/`ship_dow`/`ship_month`. Confirm it isn't hiding the same (or different) issues.

In [13]:
test_order = pd.to_datetime(test["order_date"], errors="coerce")
test_ship = pd.to_datetime(test["ship_date"], errors="coerce")

print("test order_date/ship_date unparseable:", test_order.isnull().sum(), test_ship.isnull().sum())
print("test ship_date < order_date:", (test_ship < test_order).sum())
print("test dates in the future:", (test_ship > pd.Timestamp.today().normalize()).sum())

test_dow_mismatch = (test_ship.dt.dayofweek != test["ship_dow"]) & test_ship.notnull()
test_month_mismatch = (test_ship.dt.month != test["ship_month"]) & test_ship.notnull()
print("test ship_dow disagrees with ship_date:", test_dow_mismatch.sum())
print("test ship_month disagrees with ship_date:", test_month_mismatch.sum())
print("test ship_dow out of [0,6]:", (~test["ship_dow"].between(0, 6)).sum())
print("test ship_month out of [1,12]:", (~test["ship_month"].between(1, 12)).sum())

test order_date/ship_date unparseable: 0 0
test ship_date < order_date: 0
test dates in the future: 0
test ship_dow disagrees with ship_date: 0
test ship_month disagrees with ship_date: 0
test ship_dow out of [0,6]: 0
test ship_month out of [1,12]: 0


## Summary of findings

- **Duplicate `shipment_id` rows in `train.csv`**: 81 ids each appear twice (162 rows total), and every one of those pairs is a byte-for-byte exact duplicate row. This matches the data dictionary's note — de-dupe to one row per id before modeling/prediction.
- **Duplicate `shipment_id` rows in `test.csv`**: the same issue shows up there too, separately — 6 ids each appear twice, also exact duplicates.
- **`shipment_id` overlap between `train.csv` and `test.csv`**: 63 ids appear in both files with identical values. Worth deciding how to handle before treating the split as fully disjoint.
- **`shipment_id` format**: otherwise fully well-formed — constant `SHP` prefix, constant middle segment (`42`), constant 7-digit numeric suffix, no case variation, no stray whitespace, no nulls. No malformed ids found.
- **`order_date` / `ship_date`**: consistently `YYYY-MM-DD` strings (fixed length, no stray whitespace, no mixed formats), no null or unparseable values, no `ship_date` earlier than `order_date`, no future-dated rows. Order-to-ship lag is 0-7 days. All of this holds in `test.csv` too.
- **`ship_dow` / `ship_month`**: values are in-range (0-6, 1-12) and agree exactly with the day-of-week/month recomputed from `ship_date` — no inconsistencies found, in either `train.csv` or `test.csv`.

Net: the only real data quality issues found in the identifiers/dates columns are the duplicate rows (in both files) and the train/test id overlap above — the date fields themselves are clean.

No rows were dropped or modified in this notebook — cleaning happens in a later step.

## Rows flagged for the team

Every row-drop candidate found above gets saved to one file, `flagged_rows.csv`, with a
reason attached. Nothing gets dropped here — a later shared step reads every column
group's `flagged_rows.csv`, combines them, and drops rows once. See
`../../../data/row_flags/README.md` for the full convention.

- **Exact duplicate `shipment_id` rows within `train.csv`**: 81 ids each have one extra
  identical copy.
- **Exact duplicate `shipment_id` rows within `test.csv`**: same issue, separately — 6
  ids each have one extra identical copy.
- **`shipment_id` overlap between `train.csv` and `test.csv`**: 63 ids show up in both
  files. This flags *both* rows (the one in train and the one in test) so whoever makes
  the call has the full picture.

In [14]:
import os

FLAGS_DIR = "../../../data/row_flags"
os.makedirs(FLAGS_DIR, exist_ok=True)

flags = []

# Exact duplicates within train.csv: flag every copy after the first. shipment_id
# alone can't say *which* copy to drop later (both copies share it), so we record
# the row_index too.
train_dedupe_mask = train.duplicated(subset="shipment_id", keep="first")
train_dedupe_flags = train.loc[train_dedupe_mask, ["shipment_id"]].reset_index().rename(columns={"index": "row_index"})
train_dedupe_flags["file"] = "train"
train_dedupe_flags["reason"] = "exact_duplicate_shipment_id"
flags.append(train_dedupe_flags)

# Same thing, but within test.csv -- it has its own 6 duplicate ids, separate from
# the train.csv duplicates above.
test_dedupe_mask = test.duplicated(subset="shipment_id", keep="first")
test_dedupe_flags = test.loc[test_dedupe_mask, ["shipment_id"]].reset_index().rename(columns={"index": "row_index"})
test_dedupe_flags["file"] = "test"
test_dedupe_flags["reason"] = "exact_duplicate_shipment_id"
flags.append(test_dedupe_flags)

# Train/test overlap: flag both sides (the train row AND the matching test row) so
# whoever decides what to do about it can see both.
train_overlap = train.loc[train["shipment_id"].isin(overlap_ids), ["shipment_id"]].reset_index().rename(columns={"index": "row_index"})
train_overlap["file"] = "train"
train_overlap["reason"] = "train_test_id_overlap"
flags.append(train_overlap)

test_overlap = test.loc[test["shipment_id"].isin(overlap_ids), ["shipment_id"]].reset_index().rename(columns={"index": "row_index"})
test_overlap["file"] = "test"
test_overlap["reason"] = "train_test_id_overlap"
flags.append(test_overlap)

flagged_rows = pd.concat(flags, ignore_index=True)[["file", "row_index", "shipment_id", "reason"]]
flagged_rows.to_csv(f"{FLAGS_DIR}/flagged_rows.csv", index=False)
print("flagged_rows saved:", len(flagged_rows))
flagged_rows.groupby(["file", "reason"]).size()

flagged_rows saved: 213


file   reason                     
test   exact_duplicate_shipment_id     6
       train_test_id_overlap          63
train  exact_duplicate_shipment_id    81
       train_test_id_overlap          63
dtype: int64